# 01 — Exploratory Data Analysis

Ceilometer backscatter dataset — cloud detection (binary: `true` = cloudy, `false` = clear).

Runs both locally and on Google Colab — the dataset path comes from `configs/config.yaml`
(`dataset.path`). CPU-only, no GPU required.

In [ ]:
import os, sys

IS_COLAB = os.path.exists('/content')

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    repo_path = '/content/Cloud_detection_project'
    if not os.path.exists(repo_path):
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        os.system(f'git clone https://{token}@github.com/caMatt99/Cloud_detection_project.git {repo_path}')

    # Sincronizza il repository con l'ultimo codice da GitHub
    os.system('cd /content/Cloud_detection_project && git pull origin main')
    print("✓ Repository sincronizzato con GitHub")

    sys.path.insert(0, f'{repo_path}/src')
    os.chdir(repo_path)
else:
    project_root = '/Users/matteo/Desktop/Cloud_detection_project'
    os.chdir(project_root)

    # Sincronizza il repository con l'ultimo codice da GitHub
    os.system('git pull origin main')
    print("✓ Repository sincronizzato con GitHub")

    sys.path.insert(0, f'{project_root}/src')

print(f"✓ Working dir: {os.getcwd()}")

## 1. Setup

In [ ]:
import os
import random

import numpy as np
import matplotlib.pyplot as plt
import yaml
from PIL import Image
import torch
import sklearn.metrics  # used later when comparing model predictions against these EDA baselines

from dataset import get_transforms, Cutout

random.seed(42)
np.random.seed(42)

In [ ]:
config_path = os.path.join(os.getcwd(), 'configs', 'config.yaml')

if not os.path.exists(config_path):
    raise FileNotFoundError(f"Config non trovato: {config_path}")

with open(config_path) as f:
    cfg = yaml.safe_load(f)

env = "colab" if os.path.exists('/content') else "local"
dataset_path = cfg["dataset"]["path"][env]
image_size = cfg["dataset"]["image_size"]

print(f"✓ Config caricato da: {config_path}")
print(f"✓ Dataset path ({env}): {dataset_path}")

In [ ]:
SPLITS = ["train", "val", "test"]
CLASSES = ["false", "true"]  # alphabetical order == ImageFolder label order (false=0, true=1)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset non trovato: {dataset_path}")

print("Verifica struttura dataset:")
for split in SPLITS:
    split_path = os.path.join(dataset_path, split)
    ok = os.path.isdir(split_path)
    print(f"  {split}/  {'✓' if ok else '✗ MANCANTE'}")
    if ok:
        for cls in CLASSES:
            cls_path = os.path.join(split_path, cls)
            cls_ok = os.path.isdir(cls_path)
            print(f"    {cls}/  {'✓' if cls_ok else '✗ MANCANTE'}")

## 2. Dataset Overview

In [ ]:
def list_images(split, cls):
    path = os.path.join(dataset_path, split, cls)
    return [f for f in os.listdir(path) if not f.startswith('.')]

counts = {split: {cls: len(list_images(split, cls)) for cls in CLASSES} for split in SPLITS}

total_per_split = {split: sum(counts[split].values()) for split in SPLITS}
total_images = sum(total_per_split.values())

expected = {"train": 770, "val": 328, "test": 470}  # numeri riportati nel paper

print("Numero immagini per split:")
for split in SPLITS:
    match = "✓" if total_per_split[split] == expected[split] else "✗"
    print(f"  {split:5s}: {total_per_split[split]:4d}  (atteso {expected[split]})  {match}")

print(f"\nTotale immagini: {total_images}")
print(f"Split ratio (train/val/test): "
      f"{total_per_split['train']/total_images:.1%} / "
      f"{total_per_split['val']/total_images:.1%} / "
      f"{total_per_split['test']/total_images:.1%}")

In [ ]:
print("Class balance per split (true=cloudy, false=clear):\n")
for split in SPLITS:
    n_true, n_false = counts[split]["true"], counts[split]["false"]
    ratio = n_true / n_false if n_false else float("nan")
    print(f"  {split:5s} -> true: {n_true:4d}   false: {n_false:4d}   (true/false ratio: {ratio:.2f})")

# Class weights (inverse frequency) sul train set, per un'eventuale loss ponderata.
# weight_c = N_totale / (n_classi * n_c) — normalizzato cosi' il peso medio e' 1.0
n_true, n_false = counts["train"]["true"], counts["train"]["false"]
n_total = n_true + n_false
w_false = n_total / (2 * n_false)
w_true = n_total / (2 * n_true)

print(f"\nClass weights (inverse frequency, train set):")
print(f"  false: {w_false:.3f}")
print(f"  true:  {w_true:.3f}")

imbalance_ratio = max(n_true, n_false) / min(n_true, n_false)
print(f"\nImbalance ratio (classe maggioritaria/minoritaria): {imbalance_ratio:.2f}")
print("Il paper non menziona sbilanciamento, ma il train set e' ~2:1 (true:false) —"
      " da tenere in considerazione per class weighting o sampling."
      if imbalance_ratio > 1.5 else
      "Dataset ragionevolmente bilanciato.")

## 3. Image Statistics

In [ ]:
def load_rgb(path):
    return np.array(Image.open(path).convert("RGB"), dtype=np.float32) / 255.0

# Campiona 10 immagini random dal train (mix di entrambe le classi)
sample_paths = []
for cls in CLASSES:
    cls_dir = os.path.join(dataset_path, "train", cls)
    files = list_images("train", cls)
    chosen = random.sample(files, min(5, len(files)))
    sample_paths += [os.path.join(cls_dir, f) for f in chosen]

sample_images = [load_rgb(p) for p in sample_paths]
print(f"Campionate {len(sample_images)} immagini da train/ per le statistiche sui pixel.")

# Mean / std / min / max per canale RGB, sull'intero campione
stacked = np.concatenate([im.reshape(-1, 3) for im in sample_images], axis=0)  # (N_pixels, 3)

channel_mean = stacked.mean(axis=0)
channel_std = stacked.std(axis=0)
channel_min = stacked.min(axis=0)
channel_max = stacked.max(axis=0)

for i, ch in enumerate(["R", "G", "B"]):
    print(f"  {ch}: mean={channel_mean[i]:.3f}  std={channel_std[i]:.3f}  "
          f"min={channel_min[i]:.3f}  max={channel_max[i]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
colors = ["tab:red", "tab:green", "tab:blue"]
for i, (ch, color) in enumerate(zip(["R", "G", "B"], colors)):
    ax.hist(stacked[:, i], bins=50, alpha=0.5, label=ch, color=color)
ax.set_title("Istogramma pixel intensity (10 immagini campionate, train)")
ax.set_xlabel("Intensita' pixel (0-1)")
ax.set_ylabel("Conteggio")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
imagenet_mean = np.array([0.485, 0.456, 0.406])
imagenet_std = np.array([0.229, 0.224, 0.225])

print("Confronto con statistiche ImageNet (usate per la normalizzazione in src/dataset.py):\n")
for i, ch in enumerate(["R", "G", "B"]):
    print(f"  {ch}: dataset={channel_mean[i]:.3f}±{channel_std[i]:.3f}   "
          f"ImageNet={imagenet_mean[i]:.3f}±{imagenet_std[i]:.3f}")

# R ~= G ~= B? Il ceilometer produce immagini in scala di grigi replicate sui 3 canali,
# oppure ha una vera componente colore (es. colormap applicata al backscatter)?
rg_diff = np.abs(stacked[:, 0] - stacked[:, 1]).mean()
gb_diff = np.abs(stacked[:, 1] - stacked[:, 2]).mean()
rb_diff = np.abs(stacked[:, 0] - stacked[:, 2]).mean()

print(f"\nDifferenza media assoluta tra canali: R-G={rg_diff:.4f}  G-B={gb_diff:.4f}  R-B={rb_diff:.4f}")
if max(rg_diff, gb_diff, rb_diff) < 0.01:
    print("-> I canali sono quasi identici: le immagini sono di fatto grayscale (R≈G≈B).")
else:
    print("-> I canali differiscono in modo non trascurabile: e' presente una vera componente colore"
          " (probabile colormap applicata al segnale di backscatter).")

## 4. Visual Inspection

In [ ]:
# Griglia 2x3: 3 immagini cloud=true (label 1) + 3 immagini cloud=false (label 0)
# (una vera griglia 3x3 richiederebbe 9 immagini/3 classi: qui le classi sono solo 2,
#  quindi 2 righe x 3 colonne copre le "3x true / 3x false" richieste).
fig, axes = plt.subplots(2, 3, figsize=(10, 8))

for row, cls in enumerate(["true", "false"]):
    files = list_images("train", cls)
    chosen = random.sample(files, 3)
    for col, fname in enumerate(chosen):
        img = Image.open(os.path.join(dataset_path, "train", cls, fname)).convert("RGB")
        axes[row, col].imshow(img)
        axes[row, col].set_title(f"{cls} (label={CLASSES.index(cls)})")
        axes[row, col].axis("off")

plt.suptitle("Campioni dal train set")
plt.tight_layout()
plt.show()

In [ ]:
# Verifica della distorsione introdotta da Resize(224x224) — le immagini originali
# sono 150x1000 (larghezza x altezza): un profilo verticale molto stretto e alto.
train_transform, eval_transform = get_transforms(image_size)

sample_path = os.path.join(dataset_path, "train", "true", list_images("train", "true")[0])
original = Image.open(sample_path).convert("RGB")
resized = original.resize((image_size, image_size))

fig, axes = plt.subplots(1, 2, figsize=(8, 5))
axes[0].imshow(original)
axes[0].set_title(f"Originale {original.size[0]}x{original.size[1]} (w x h)")
axes[0].axis("off")

axes[1].imshow(resized)
axes[1].set_title(f"Resize({image_size}x{image_size})")
axes[1].axis("off")

plt.tight_layout()
plt.show()

print(f"Aspect ratio originale: {original.size[0]/original.size[1]:.3f} (larghezza/altezza)")
print(f"Aspect ratio dopo resize: 1.000")
print("Nota: il resize a un quadrato distorce fortemente l'immagine (schiaccia l'asse verticale"
      " di un fattore ~6.7x), esattamente come descritto in src/dataset.py — approccio ereditato"
      " dal paper originale, non corretto per l'aspect ratio.")

## 5. Comparison with Paper

Il baseline del paper (ResNet50) raggiunge **89.57% accuracy** sul test set.

Architetture da testare/confrontare in questo progetto (vedi `src/models.py`):
- `resnet50` — baseline del paper
- `convnext_base` — CNN moderna (2022), non testata nel paper originale
- `swin_base` — Vision Transformer gerarchico
- `vit` — Vision Transformer (gia' testato nel paper)

**Nota sulla dimensione del dataset**: 770 immagini di train sono poche per il fine-tuning di
backbone pretrenati su ImageNet (soprattutto per architetture transformer-based come ViT/Swin,
piu' data-hungry delle CNN) — alto rischio di overfitting. Da qui le contromisure gia' presenti
nel progetto: data augmentation (RandAugment, Cutout, sezione 6), gradual unfreezing e dropout
modulare (`src/models.py::get_unfrozen_params`, `apply_dropout`).

**Riferimento**: Chisari et al., *"Cloud Detection Challenge – Methods and Results"*, IEEE Access 2025.

## 6. Data Augmentation Preview

In [ ]:
from torchvision import transforms

# Immagine di riferimento, ridimensionata come nella pipeline reale
base_img = Image.open(sample_path).convert("RGB").resize((image_size, image_size))

# Trasformazioni applicate singolarmente per ispezione visiva.
# p=1.0 sul flip cosi' l'effetto e' sempre visibile (in training e' p=0.5).
hflip = transforms.RandomHorizontalFlip(p=1.0)(base_img)
randaug = transforms.RandAugment(num_ops=2, magnitude=9)(base_img)

# Cutout (src/dataset.py) opera su tensori normalizzati: qui, per chiarezza visiva,
# lo applichiamo direttamente al tensore [0,1] (senza Normalize) cosi' il buco resta nero
# invece che al valore medio della distribuzione normalizzata.
to_tensor = transforms.ToTensor()
cutout_tensor = Cutout(num_holes=1, max_h_size=32, max_w_size=32)(to_tensor(base_img).clone())
cutout_img = transforms.ToPILImage()(cutout_tensor)

fig, axes = plt.subplots(1, 4, figsize=(14, 5))
for ax, img, title in zip(
    axes,
    [base_img, hflip, randaug, cutout_img],
    ["Originale (resized)", "RandomHorizontalFlip", "RandAugment", "Cutout"],
):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

**RandomVerticalFlip avrebbe senso per i profili ceilometer?**

Probabilmente no. L'immagine e' un profilo verticale backscatter-vs-tempo: l'asse verticale
rappresenta l'altitudine (dal suolo verso l'alto), mentre l'asse orizzontale rappresenta il tempo.
Un flip verticale inverte fisicamente l'altitudine (le nubi apparirebbero vicino al suolo e
viceversa), producendo un profilo non fisico che il modello non vedrebbe mai in produzione — rischia
di introdurre segnale ingannevole piuttosto che una vera augmentation. Il flip orizzontale (inversione
dell'asse temporale) e' invece gia' usato nel paper ed e' meno problematico, anche se comunque
altera l'ordine cronologico del profilo.

## 7. Statistical Tests (opzionale)

In [ ]:
from scipy import stats

# T-test: il pixel mean (per immagine) e' diverso tra cloud=true e cloud=false?
# Usiamo l'intero train set (770 immagini, poche centinaia di ms) invece del solo
# campione da 10 immagini della sezione 3, per dare piu' potenza statistica al test.

def per_image_mean(cls, split="train"):
    values = []
    for fname in list_images(split, cls):
        img = load_rgb(os.path.join(dataset_path, split, cls, fname))
        values.append(img.mean())
    return np.array(values)

mean_true = per_image_mean("true")
mean_false = per_image_mean("false")

print(f"cloud=true  : n={len(mean_true):4d}  pixel_mean={mean_true.mean():.4f}  std={mean_true.std():.4f}")
print(f"cloud=false : n={len(mean_false):4d}  pixel_mean={mean_false.mean():.4f}  std={mean_false.std():.4f}")

# Welch's t-test (equal_var=False): non assumiamo varianze uguali tra le due classi,
# ne' dimensioni di campione uguali (515 vs 255).
t_stat, p_value_ttest = stats.ttest_ind(mean_true, mean_false, equal_var=False)

alpha = 0.05
print(f"\nWelch t-test: t={t_stat:.3f}  p-value={p_value_ttest:.2e}")
if p_value_ttest < alpha:
    print(f"-> p < {alpha}: il pixel mean e' significativamente diverso tra cloud=true e cloud=false"
          " (il canale colore porta segnale utile per la classificazione, come atteso).")
else:
    print(f"-> p >= {alpha}: nessuna differenza significativa nel pixel mean tra le due classi.")

In [ ]:
from scipy import stats

# Chi-square: la distribuzione delle classi (true/false) e' significativamente diversa
# tra train/val/test, oppure gli split sono stati creati mantenendo le stesse proporzioni?
contingency = np.array([[counts[split]["true"], counts[split]["false"]] for split in SPLITS])

print("Tabella di contingenza (righe=split, colonne=[true, false]):")
for split, row in zip(SPLITS, contingency):
    print(f"  {split:5s}: {row.tolist()}")

chi2_stat, p_value_chi2, dof, expected = stats.chi2_contingency(contingency)

print(f"\nChi-square test: chi2={chi2_stat:.3f}  dof={dof}  p-value={p_value_chi2:.3f}")
if p_value_chi2 < alpha:
    print(f"-> p < {alpha}: la distribuzione delle classi differisce significativamente tra gli split"
          " (train/val/test NON sono stratificati sulla classe).")
else:
    print(f"-> p >= {alpha}: nessuna differenza significativa — gli split mantengono la stessa"
          " proporzione true/false (stratificazione implicita o casuale ma bilanciata).")

In [ ]:
print("Conclusione:\n")
print(f"- Imbalance ratio (train, true/false): {imbalance_ratio:.2f}")
print(f"- T-test pixel mean true vs false: p={p_value_ttest:.2e} "
      f"({'differenza significativa' if p_value_ttest < alpha else 'nessuna differenza significativa'})")
print(f"- Chi-square class distribution tra split: p={p_value_chi2:.3f} "
      f"({'split NON stratificati' if p_value_chi2 < alpha else 'split stratificati/bilanciati'})")

if imbalance_ratio > 1.5:
    print(
        "\n-> Il dataset e' sbilanciato (~2:1 true:false) in modo consistente su tutti gli split "
        "(vedi test chi-square). Consigliato usare i class weights calcolati in sezione 2 "
        "(nn.CrossEntropyLoss(weight=...)) o un WeightedRandomSampler, specialmente per le "
        "architetture piu' data-hungry (ViT/Swin) allenate su un train set cosi' piccolo."
    )
else:
    print("\n-> Il dataset e' ragionevolmente bilanciato: il class weighting non e' strettamente necessario.")